In [1]:
import pandas as pd
import csv 
import numpy as np

pd.set_option("display.max_colWidth", None)


ARQUIVO_GRID = "/lakehouse/default/Files/raw_cadastro_carta/grid_cadastro_carta.csv"
ARQUIVO_BD = (
    "/lakehouse/default/Files/raw_cadastro_carta/bd_entidade_cadastro_carta.csv"
)
bd_test = pd.read_csv(
    ARQUIVO_BD,
    sep=";",
    encoding="utf-8-sig",
    engine="python",
    quotechar='"',
    doublequote=True,
    quoting=csv.QUOTE_MINIMAL,
    on_bad_lines="skip",
    dtype=str,
)

print(bd_test.columns.tolist())

StatementMeta(, a2606a6b-c4c7-4606-b3a7-badb6853bdbc, 3, Finished, Available, Finished, False)

['DATA DE ATUALIZAÇÃO', 'Nome:', 'Categoria:', 'Público alvo:', 'Área responsável:', 'ID do serviço:', 'Status do serviço:']


In [2]:
import pandas as pd
import csv
from unidecode import unidecode
import numpy as np
from datetime import datetime

pd.set_option("display.max_colWidth", None)


ARQUIVO_GRID = "/lakehouse/default/Files/raw_cadastro_carta/grid_cadastro_carta.csv"
ARQUIVO_BD = (
    "/lakehouse/default/Files/raw_cadastro_carta/bd_entidade_cadastro_carta.csv"
)


def carregar_tratar_bd():

    bd = pd.read_csv(
        ARQUIVO_BD,
        sep=";",
        encoding="utf-8-sig",
        engine="python",
        quotechar='"',
        doublequote=True,
        quoting=csv.QUOTE_MINIMAL,
        on_bad_lines="skip",
        dtype=str,
    )

    # unidades dos servicos duplicam infos,
    # ex: servico de matricula, unidade: escola a, escola b, escola c.
    # retirando as colunas e removendo duplicados na sequencia
    # bd = bd.drop(
    #     columns=["Como acessar o serviço:", "Sigla da unidade:", "Nome da Unidade:"]
    # )
    # bd = bd.drop_duplicates()

    # padronização das colunas
    bd.columns = bd.columns.str.lower().str.replace(":", "").str.replace(" ", "_")
    bd.columns = [unidecode(col) for col in bd.columns]

    bd = bd.rename(
        columns={
            "nome": "nome_do_servico",
        }
    )

    # inclusão de colunas
    bd["servico"] = "Cadastro de carta de serviço"
    bd["status_tramitacao"] = "Finalizado"

    return bd


def carregar_tratar_grid():
    grid = pd.read_csv(
        ARQUIVO_GRID,
        sep=";",
        encoding="utf-8-sig",
        engine="python",
        quotechar='"',
        doublequote=True,
        quoting=csv.QUOTE_MINIMAL,
        on_bad_lines="skip",
        dtype=str,
    )

    if "Unnamed: 15" in grid.columns:
        grid = grid.drop(columns=["Unnamed: 15"])

    # padronização das colunas
    grid.columns = grid.columns.str.lower().str.replace(":", "").str.replace(" ", "_")
    grid.columns = [unidecode(col) for col in grid.columns]

    grid = grid.rename(
        columns={
            "status": "status_tramitacao",
            "status_servico": "status_do_servico",
            "id_carta": "id_do_servico",
        }
    )

    # filtra apenas ordems em aberto
    grid_em_aberto = grid.query(
        "status_tramitacao.isin(['Em atendimento', 'Pendente'])"
    ).copy()

    # remove colunas indesejadas
    grid_em_aberto = grid_em_aberto.drop(
        columns=[
            # "unnamed_15",
            "no_da_solicitacao",
            "etapa_atual",
            "nome_aprovado",
            "executor_atual",
            "solicitante",
            "data_de_finalizacao",
        ]
    )

    return grid, grid_em_aberto


def gerar_bd_final(grid_cleaned, bd_cleaned):

    df_bi_cadastro_carta = pd.concat([grid_cleaned, bd_cleaned], axis=0)

    servicos_fora_portal = {
        "Alvará de Reforma",
        "Matricula escolar",
        "Acesso Técnicos da rede Socioassistencial",
        "Cadastro de OSC's parceiras",
    }

    # data_consolidada = se finalizado: data_finalizacao, se em aberto: data_solicitacao
    df_bi_cadastro_carta["data_consolidada"] = np.where(
        df_bi_cadastro_carta["data_de_atualizacao"].isna(),
        df_bi_cadastro_carta["data_de_solicitacao"],
        df_bi_cadastro_carta["data_de_atualizacao"],
    )

    for col in ["data_de_atualizacao", "data_de_solicitacao", "data_consolidada"]:
        df_bi_cadastro_carta[col] = pd.to_datetime(
            df_bi_cadastro_carta[col], dayfirst=True
        )
        df_bi_cadastro_carta[col] = df_bi_cadastro_carta[col].dt.date

    df_bi_cadastro_carta["sigla_area_responsavel"] = df_bi_cadastro_carta[
        "area_responsavel"
    ].str.split(" -", expand=True)[0]

    df_bi_cadastro_carta["data_de_atualizacao"] = pd.to_datetime(
        df_bi_cadastro_carta["data_de_atualizacao"]
    )
    df_bi_cadastro_carta["dias_desde_atualizacao"] = (
        pd.to_datetime(datetime.now()) - df_bi_cadastro_carta["data_de_atualizacao"]
    ).dt.days

    def categorizar_dias_atualizacao(dias):
        if dias < 30:
            return "Menor que 30 dias"
        elif dias < 60:
            return "Entre 30 e 60 dias"
        elif dias < 90:
            return "Entre 60 e 90 dias"
        elif dias < 120:
            return "Entre 90 e 120 dias"
        elif dias < 365:
            return "Entre 120 e 365 dias"
        elif pd.isna(dias):
            return "Não se aplica"
        else:
            return "Maior que 365 dias"

    df_bi_cadastro_carta["periodo_atualizacao"] = df_bi_cadastro_carta[
        "dias_desde_atualizacao"
    ].apply(categorizar_dias_atualizacao)

    # df_bi_cadastro_carta["id_do_servico"] = np.where(
    #     (df_bi_cadastro_carta["id_do_servico"].notna())
    #     & (df_bi_cadastro_carta["status_tramitacao"] != "Finalizado"),
    #     np.nan,
    #     df_bi_cadastro_carta["id_do_servico"],
    # )

    return df_bi_cadastro_carta


def gerar_grid_final(grid):
    grid["sigla_area_responsavel"] = grid["area_responsavel"].str.split(
        " - ", expand=True
    )[0]

    grid = (
        grid[ # .loc[grid["status_tramitacao"].isin(["Em atendimento", "Pendente"])][
            [
                "no_da_solicitacao",
                "status_tramitacao",
                "nome_do_servico",
                "data_de_solicitacao",
                "id_do_servico",
                "etapa_atual",
                "executor_atual",
                "solicitante",
                "categoria",
                "area_responsavel",
                "sigla_area_responsavel",
            ]
        ]
        .drop_duplicates()
        .sort_values("nome_do_servico")
    )
    return grid


bd_cleaned = carregar_tratar_bd()
grid, grid_cleaned = carregar_tratar_grid()
df_bi_cadastro_carta = gerar_bd_final(grid_cleaned, bd_cleaned)
grid = gerar_grid_final(grid)

StatementMeta(, a2606a6b-c4c7-4606-b3a7-badb6853bdbc, 4, Finished, Available, Finished, False)

In [3]:
# escrita da tabela no lh
df_bi_cadastro_carta_sdf = spark.createDataFrame(df_bi_cadastro_carta)

(
    df_bi_cadastro_carta_sdf
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_carta_servicos")
)

if len(grid) > 0:
    grid_sdf = spark.createDataFrame(grid)
    (
        grid_sdf
        .write.mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .saveAsTable("gold_carta_servicos_atualizacoes")
    )
else:
    print("Não há OS em aberto!")

StatementMeta(, a2606a6b-c4c7-4606-b3a7-badb6853bdbc, 5, Finished, Available, Finished, True)